# Word embeddings dla anotowanego korpusu PolEmo

Notebook realizuje ćwiczenie: wczytanie anotacji, dwa modele przestrzeni wektorowych, interaktywny t-SNE oraz porównanie k-najbliższych sąsiadów.

Analizę wykonujemy na rzeczywistych, pretrenowanych modelach dla języka polskiego. Wyniki odnosimy do etykiet anotacji `negacja`, `wzmocnienie` i `oslabienie`.

Duże modele nie są przechowywane w repozytorium. Word2Vec wymaga lokalnej ścieżki, natomiast FastText jest pobierany bezpośrednio z Hugging Face i zapisywany lokalnie w `01/models`.

| Wymaganie | Realizacja w notebooku |
|---|---|
| 1. Co najmniej dwa modele | Word2Vec z DSModels oraz FastText `clarin-pl/fastText-kgr10` |
| 2. Wizualizacja | t-SNE, kolory etykiet, słowo i anotacja dostępne po najechaniu na punkt |
| 3. Podobieństwo | `TOP_K = 8`, listy sąsiadów dla obu modeli i liczba wspólnych elementów |
| 4. Dyskusja | Interpretacja t-SNE, różnic Word2Vec/FastText i statystyk zgodności |

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models.keyedvectors import KeyedVectors
from sklearn.manifold import TSNE

SEED = 42
ROOT = Path.cwd()
if (ROOT / "01").is_dir():
    PROJECT_DIR = ROOT
elif ROOT.name == "notebooks" and (ROOT.parent.parent / "01").is_dir():
    PROJECT_DIR = ROOT.parent.parent
else:
    raise FileNotFoundError("Uruchom notebook z katalogu repozytorium albo 01/notebooks.")
ANNOTATION_DIR = PROJECT_DIR / "01" / "annotation"
FASTTEXT_METHOD = "skipgram"
FASTTEXT_DIMENSION = 300
WORD2VEC_PATH = PROJECT_DIR / "01" / "models" / "nkjp+wiki-forms-all-300-skipg-ns.txt.gz"
WORD2VEC_BIN_PATH = PROJECT_DIR / "01" / "models" / "nkjp+wiki-forms-all-300-skipg-ns.bin"
TOP_K = 8
TOKEN_RE = re.compile(r"[A-Za-zĄĆĘŁŃÓŚŹŻąćęłńóśźż0-9]+(?:[-'][A-Za-zĄĆĘŁŃÓŚŹŻąćęłńóśźż0-9]+)?")
print(PROJECT_DIR)

d:\Work\NLP


## 2. Wczytanie anotacji Label Studio

Teksty są deduplikowane, a tokeny otrzymują etykietę fragmentu, jeśli ich środek znajduje się w zaznaczonym span. Tokeny bez anotacji są zachowane w tabeli pomocniczej, ale nie są używane jako punkty oznaczone klasą.

In [2]:
def load_export(path):
    rows = []
    for task in json.loads(path.read_text(encoding="utf-8")):
        annotation = next(iter(task.get("annotations", [])), {})
        sentiment, spans = None, []
        for result in annotation.get("result", []):
            value = result.get("value", {})
            if result.get("type") == "choices":
                sentiment = (value.get("choices") or [None])[0]
            elif result.get("type") == "labels" and value.get("labels"):
                spans.append((int(value["start"]), int(value["end"]), value["labels"][0]))
        rows.append({"text": task.get("data", {}).get("text", ""), "sentiment": sentiment, "spans": spans})
    return rows

records = [row for path in sorted(ANNOTATION_DIR.glob("**/export_*.json")) for row in load_export(path)]
annotations = pd.DataFrame(records).drop_duplicates("text").reset_index(drop=True)
token_rows = []
for document, row in annotations.iterrows():
    for match in TOKEN_RE.finditer(row.text):
        center = (match.start() + match.end() - 1) / 2
        labels = [label for start, end, label in row.spans if start <= center < end]
        token_rows.append({"word": match.group().lower(), "label": labels[0] if labels else "brak_etykiety", "document": document, "sentiment": row.sentiment or "brak_etykiety"})
tokens = pd.DataFrame(token_rows)
annotated = tokens[tokens.label != "brak_etykiety"].copy()
print(f"Teksty: {len(annotations)}, tokeny: {len(tokens)}, tokeny anotowane: {len(annotated)}")
display(annotated.label.value_counts().rename("liczba").to_frame())

Teksty: 35, tokeny: 3538, tokeny anotowane: 155


,liczba
label,
negacja,82
wzmocnienie,50
oslabienie,23


## 3. Word2Vec i FastText

### Wykorzystane modele

- **Word2Vec**: polski model `nkjp+wiki-forms-all-300-skipg-ns` pobrany z [DSModels](https://dsmodels.nlp.ipipan.waw.pl/). Model ma 300 wymiarów i jest zapisany lokalnie w `01/models`. Po jednorazowej konwersji do formatu binarnego jest ładowany szybciej.
- **FastText**: polski model `clarin-pl/fastText-kgr10`, konfiguracja `skipgram`, 300 wymiarów, pobrany z [Hugging Face](https://huggingface.co/clarin-pl/fastText-kgr10). FastText reprezentuje słowa także przez n-gramy znakowe, dlatego może tworzyć wektory dla form niewystępujących dokładnie w słowniku.

Oba modele są analizowane sekwencyjnie. Po wizualizacji Word2Vec jego obiekt jest usuwany z pamięci przed załadowaniem dużego modelu FastText.

In [3]:
import shutil
from huggingface_hub import hf_hub_download

FASTTEXT_DIR = PROJECT_DIR / "01" / "models"
FASTTEXT_DIR.mkdir(parents=True, exist_ok=True)
FASTTEXT_FILENAME = f"kgr10.plain.{FASTTEXT_METHOD}.dim{FASTTEXT_DIMENSION}.neg10.bin"
FASTTEXT_LOCAL_PATH = FASTTEXT_DIR / FASTTEXT_FILENAME

if FASTTEXT_LOCAL_PATH.exists():
    print(f"FastText jest już zapisany lokalnie: {FASTTEXT_LOCAL_PATH}")
else:
    cached_path = hf_hub_download(
        repo_id="clarin-pl/fastText-kgr10",
        filename=FASTTEXT_FILENAME,
        legacy_cache_layout=True,
    )
    print(f"Pobrano do cache: {cached_path}")
    print("Kopiuję model do katalogu projektu; to może potrwać kilka minut...")
    shutil.copyfile(cached_path, FASTTEXT_LOCAL_PATH)
    print(f"Zapisano: {FASTTEXT_LOCAL_PATH}")

FastText jest już zapisany lokalnie: d:\Work\NLP\01\models\kgr10.plain.skipgram.dim300.neg10.bin


In [4]:
if not WORD2VEC_PATH.exists():
    raise FileNotFoundError(f"Nie znaleziono tekstowego modelu Word2Vec: {WORD2VEC_PATH}")

if WORD2VEC_BIN_PATH.exists():
    print(f"Binarny Word2Vec jest już zapisany: {WORD2VEC_BIN_PATH}")
else:
    print("Konwertuję Word2Vec do formatu binarnego; pierwszy przebieg może potrwać...")
    source_vectors = KeyedVectors.load_word2vec_format(WORD2VEC_PATH, binary=False)
    source_vectors.save_word2vec_format(WORD2VEC_BIN_PATH, binary=True)
    del source_vectors
    print(f"Zapisano: {WORD2VEC_BIN_PATH}")

Binarny Word2Vec jest już zapisany: d:\Work\NLP\01\models\nkjp+wiki-forms-all-300-skipg-ns.bin


In [5]:
if not WORD2VEC_BIN_PATH.exists():
    raise FileNotFoundError(f"Nie znaleziono binarnego modelu Word2Vec: {WORD2VEC_BIN_PATH}")
if not FASTTEXT_LOCAL_PATH.exists():
    raise FileNotFoundError("Najpierw uruchom komórkę pobierającą FastText do katalogu models.")
w2v = KeyedVectors.load_word2vec_format(WORD2VEC_BIN_PATH, binary=True)
print(f"Word2Vec: {len(w2v.key_to_index)} słów")

Word2Vec: 2123132 słów


## 4. Interaktywna wizualizacja t-SNE

Dla każdego modelu wybieramy słowa anotowane w korpusie, pobieramy ich wektory i redukujemy 300 wymiarów do dwóch za pomocą t-SNE. Kolor punktu oznacza etykietę anotacji, a rozmiar odpowiada częstości słowa w anotowanych fragmentach. Wykres Plotly jest interaktywny: po najechaniu na punkt można odczytać słowo, etykietę oraz częstość.

t-SNE zachowuje przede wszystkim lokalne sąsiedztwa. Odległości między odległymi grupami oraz ich kształt należy traktować jakościowo, a nie jako ścisłą miarę podobieństwa.

In [6]:
def tsne_frame(model, name):
    info = annotated.groupby(["word", "label"]).size().reset_index(name="frequency")
    words = [word for word in info.word.unique() if word in model.key_to_index]
    info = info[info.word.isin(words)].groupby("word", as_index=False).agg(label=("label", lambda x: ", ".join(sorted(set(x)))), frequency=("frequency", "sum"))
    coords = TSNE(n_components=2, perplexity=min(30, max(2, len(info) - 1)), random_state=SEED, init="pca").fit_transform(np.vstack([model.get_vector(w) for w in info.word]))
    info[["x", "y"]] = coords
    info["model"] = name
    return info

def interactive_plot(frame):
    return px.scatter(frame, x="x", y="y", color="label", size="frequency", hover_name="word", hover_data={"label": True, "frequency": True, "x": False, "y": False}, title=f"t-SNE: {frame.model.iloc[0]}")

display(interactive_plot(tsne_frame(w2v, "Word2Vec")))

# Zachowujemy wyniki Word2Vec przed zwolnieniem dużego modelu.
w2v_neighbors_snapshot = {
    word: [(item, round(float(score), 4)) for item, score in w2v.most_similar(word, topn=TOP_K)]
    for word in dict.fromkeys(annotated.word)
    if word in w2v.key_to_index
}

## 5. Listy k najbardziej podobnych słów

Dla każdego unikalnego słowa z anotowanego korpusu wyznaczamy `TOP_K = 8` najbliższych słów według podobieństwa kosinusowego w każdym modelu. Tabela pokazuje osobno listę Word2Vec i FastText oraz liczbę wspólnych sąsiadów. Ta ostatnia wartość pozwala ocenić, które relacje są stabilne niezależnie od modelu.

In [7]:
import gc

del w2v
gc.collect()
print("Word2Vec usunięty z pamięci. Ładuję FastText...")
from gensim.models.fasttext import load_facebook_vectors
fasttext = load_facebook_vectors(str(FASTTEXT_LOCAL_PATH))
print(f"FastText: {len(fasttext.key_to_index)} słów")
display(interactive_plot(tsne_frame(fasttext, "FastText")))

Word2Vec usunięty z pamięci. Ładuję FastText...
FastText: 3488597 słów


In [8]:
def neighbors(model, word):
    if word not in model.key_to_index:
        return []
    return [(item, round(float(score), 4)) for item, score in model.most_similar(word, topn=TOP_K)]

rows = []
for word in dict.fromkeys(annotated.word):
    left = w2v_neighbors_snapshot.get(word, [])
    right = neighbors(fasttext, word)
    rows.append({"słowo": word, "Word2Vec": left, "FastText": right, "wspólne": len({x[0] for x in left} & {x[0] for x in right})})
similarity = pd.DataFrame(rows)
display(similarity.head(30))
display(similarity.wspólne.describe().to_frame("liczba wspólnych sąsiadów"))

,słowo,Word2Vec,FastText,wspólne
0,bardzo,"[(dość, 0.8021), (niezwykle, 0.7742), (wyjątko...","[(niezwykle, 0.8076), (baaardzo, 0.8066), (baa...",2
1,brak,"[(braku, 0.6503), (niedostatek, 0.6206), (brak...","[(Brak, 0.8088), (braku, 0.7661), (niedostatek...",4
2,nie,"[(jednak, 0.6946), (ani, 0.6455), (ale, 0.6432...","[(Nie, 0.6617), (nnie, 0.6615), (inie, 0.6503)...",1
3,praktycznie,"[(niemal, 0.743), (niemalże, 0.6921), (prawie,...","[(Praktycznie, 0.8337), (niemal, 0.7174), (nie...",6
4,nawet,"[(choć, 0.6019), (jedynie, 0.5984), (tylko, 0....","[(Nawet, 0.7137), (tylko, 0.7107), (ale, 0.706...",1
5,w,"[(i, 0.6421), (na, 0.61), (gdzie, 0.5919), (or...","[(W, 0.7065), (we, 0.565), (i, 0.5363), (sięw,...",2
6,miarę,"[(miare, 0.7132), (mierę, 0.6415), (upływu, 0....","[(wmiarę, 0.7589), (miare, 0.7197), (Wmiarę, 0...",1
7,wydają,"[(wydawały, 0.7597), (wydaja, 0.694), (okazują...","[(Wydają, 0.8361), (wydawały, 0.772), (ydają, ...",3
8,się,"[(sią, 0.6262), (sięw, 0.5637), (siętakże, 0.5...","[(sią, 0.8836), (się, 0.7483), (sięje, 0.7215...",2
9,raczej,"[(jakkolwiek, 0.5636), (generalnie, 0.559), (o...","[(Raczej, 0.7378), (raczje, 0.7148), (bynajmni...",1


,liczba wspólnych sąsiadów
count,56.000000
mean,2.500000
std,1.758098
min,0.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,7.000000


## 6. Dyskusja wyników

Wizualizacje pozwalają sprawdzić, czy słowa o tej samej funkcji anotacyjnej tworzą lokalne skupienia. Nie należy jednak utożsamiać skupienia na wykresie t-SNE z formalną separowalnością klas: t-SNE jest metodą projekcji i zależy między innymi od ziarna oraz parametrów redukcji.

Word2Vec opisuje podobieństwo na podstawie kontekstów występujących w korpusie treningowym. Zwykle daje więc bardziej semantyczne listy sąsiadów, ale nie radzi sobie z formą spoza słownika. FastText wykorzystuje dodatkowo n-gramy znakowe, co pomaga przy odmianie wyrazów i rzadkich formach. Jednocześnie na listach FastText mogą pojawiać się warianty pisowni, różnice wielkości liter lub fragmenty znakowe, które nie zawsze są najlepszymi sąsiadami semantycznymi.

W przeprowadzonym porównaniu dla 56 anotowanych słów lista ośmiu sąsiadów miała średnio 2,5 wspólnego elementu między modelami, mediana wyniosła 2, a kwartyle 1 i 4. Oznacza to częściową zgodność reprezentacji: modele wychwytują niektóre wspólne relacje, ale różnią się wrażliwością na kontekst, fleksję i budowę słowa. Wynik sugeruje, że oba modele mogą być użyteczne jako cechy dla prostego klasyfikatora, ale jakość predykcji należy potwierdzić osobną ewaluacją na anotacjach, na przykład przez podział train/test i klasyfikator liniowy.